[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/26_Semantic_Layer_Basics.ipynb)

# Semantic Layer Basics: Putting the Knowledge Graph, Ontology, and Mappings Together

## Overview

This lesson connects three things you have already met — a knowledge graph, an ontology, and RDF export — into one minimal *semantic layer*: a knowledge graph whose types, relationships, and properties are **explicitly mapped** to ontology terms, so the resulting RDF can be queried with SPARQL against a shared vocabulary.

**Documentation**: [API Reference](https://semantica.readthedocs.io/concepts/)

### 🎯 Learning Objectives

- Build a small knowledge graph with `GraphBuilder`
- Generate a starter ontology from the graph with `OntologyGenerator`
- Write **explicit** entity-type, relationship-type, and property mappings to ontology terms
- Produce ontology-aligned RDF and store it with `TripletStore`
- Answer a business question with one small SPARQL query

### 📚 Prerequisites

- [07_Building_Knowledge_Graphs.ipynb](./07_Building_Knowledge_Graphs.ipynb) — graphs from entities and relationships
- [14_Ontology.ipynb](./14_Ontology.ipynb) — ontology generation
- [20_Triplet_Store.ipynb](./20_Triplet_Store.ipynb) — triplet store backends

> [!NOTE]
> **Teaching mappings vs. governed mappings.** The mappings in this lesson are a demo: they live in a Python dict and are derived from a generated ontology. A production semantic layer uses governed identifiers, hand-designed ontologies, explicit source mappings, validation (SHACL), provenance, and versioning — that workflow is covered in [Advanced: Manual Ontology + Snowflake Mapping](../advanced/13_Manual_Ontology_Snowflake_Mapping.ipynb).

## Installation

The triplet-store step uses the embedded Oxigraph backend, so install with that extra:

```bash
pip install "semantica[tripletstore-oxigraph]"
```

---

## Step 1: Build a Knowledge Graph

Start from a small, explicit set of entities and relationships — two people, an organization, and a project.


In [ ]:
!pip install "semantica[tripletstore-oxigraph]"


In [ ]:
from semantica.kg import GraphBuilder

entities = [
    {"id": "e1", "type": "Person", "name": "Alice", "properties": {"age": 30, "role": "Engineer"}},
    {"id": "e2", "type": "Person", "name": "Bob", "properties": {"age": 35, "role": "Manager"}},
    {"id": "e3", "type": "Organization", "name": "Tech Corp", "properties": {"founded": 2010}},
    {"id": "e4", "type": "Project", "name": "Project Alpha", "properties": {"status": "active"}},
]

relationships = [
    {"source": "e1", "target": "e2", "type": "reports_to", "properties": {}},
    {"source": "e1", "target": "e3", "type": "works_for", "properties": {}},
    {"source": "e2", "target": "e3", "type": "works_for", "properties": {}},
    {"source": "e1", "target": "e4", "type": "works_on", "properties": {}},
]

builder = GraphBuilder()
knowledge_graph = builder.build({"entities": entities, "relationships": relationships})

id_to_name = {entity["id"]: entity["name"] for entity in entities}

print(f"Entities ({len(knowledge_graph['entities'])}):")
for entity in knowledge_graph["entities"]:
    print(f"  {entity['id']}: {entity['name']} ({entity['type']}) {entity['properties']}")

print(f"\nRelationships ({len(knowledge_graph['relationships'])}):")
for relationship in knowledge_graph["relationships"]:
    print(f"  {id_to_name[relationship['source']]} "
          f"--{relationship['type']}--> {id_to_name[relationship['target']]}")

## Step 2: Generate a Starter Ontology

`OntologyGenerator` infers OWL classes and properties from the graph. Two settings matter here:

- `base_uri` puts every generated term in *your* namespace
- `min_occurrences=1` includes classes that occur only once (the default of 2 would drop `Organization` and `Project` from this tiny demo graph)

Note that the generator normalizes names: the relationship type `works_for` becomes the ontology property `worksFor`. That is exactly why the next step maps terms **explicitly** instead of matching names.


In [ ]:
from semantica.ontology import OntologyGenerator

BASE_URI = "https://example.org/company/"

generator = OntologyGenerator(base_uri=BASE_URI, min_occurrences=1)
ontology = generator.generate_from_graph(knowledge_graph)

print("Classes:")
for ontology_class in ontology["classes"]:
    print(f"  {ontology_class['name']:<14} {ontology_class['uri']}")

print("\nObject properties:")
for prop in ontology["properties"]:
    if prop["type"] == "object":
        print(f"  {prop['name']:<14} {prop['uri']}  "
              f"(domain={prop['domain']}, range={prop['range']})")

assert len(ontology["classes"]) == 3

## Step 3: Map the Graph to Ontology Terms

The heart of a semantic layer is the mapping contract: which source type, relationship, and property corresponds to which ontology term.

- **Entity types** and **relationship types**: each generated class/property records the source name it was inferred from (`metadata["inferred_from"]`), so the mapping is read off the ontology itself — no fragile name matching between `works_for` and `worksFor`.
- **Properties**: the entity's `name` maps to the ontology's generated `name` datatype property; the remaining property keys (`age`, `role`, `founded`, `status`) are mapped to terms minted explicitly in our namespace.


In [ ]:
entity_type_mappings = {
    ontology_class["metadata"]["inferred_from"]: ontology_class["uri"]
    for ontology_class in ontology["classes"]
}

relationship_type_mappings = {
    prop["metadata"]["inferred_from"]: prop["uri"]
    for prop in ontology["properties"]
    if prop["type"] == "object"
}

datatype_property_uris = {
    prop["metadata"]["inferred_from"]: prop["uri"]
    for prop in ontology["properties"]
    if prop["type"] != "object"
}

property_mappings = {
    "name": datatype_property_uris["name"],
    "age": BASE_URI + "age",
    "role": BASE_URI + "role",
    "founded": BASE_URI + "founded",
    "status": BASE_URI + "status",
}

semantic_layer = {
    "graph": knowledge_graph,
    "ontology": ontology,
    "mappings": {
        "entity_type_mappings": entity_type_mappings,
        "relationship_type_mappings": relationship_type_mappings,
        "property_mappings": property_mappings,
    },
}

for mapping_name, mapping in semantic_layer["mappings"].items():
    print(f"{mapping_name}:")
    for source, target in mapping.items():
        print(f"  {source:<12} -> {target}")

# Every type and relationship in the graph must have an ontology term
assert set(entity_type_mappings) == {entity["type"] for entity in entities}
assert set(relationship_type_mappings) == {rel["type"] for rel in relationships}

## Step 4: Apply the Mappings

Applying the semantic layer means rewriting the graph so every type, relationship, and property key is an ontology term. This *aligned* graph — not the original one — is what gets exported and stored.


In [ ]:
aligned_graph = {
    "entities": [
        {
            **entity,
            "type": entity_type_mappings[entity["type"]],
            "properties": {
                property_mappings["name"]: entity["name"],
                **{
                    property_mappings[key]: value
                    for key, value in entity["properties"].items()
                },
            },
        }
        for entity in knowledge_graph["entities"]
    ],
    "relationships": [
        {**rel, "type": relationship_type_mappings[rel["type"]]}
        for rel in knowledge_graph["relationships"]
    ],
}

print("Aligned entity sample:")
sample = aligned_graph["entities"][0]
print(f"  id:   {sample['id']}")
print(f"  type: {sample['type']}")
for key, value in sample["properties"].items():
    print(f"  {key} = {value}")

print("\nAligned relationship sample:")
print(f"  {aligned_graph['relationships'][0]['type']}")

## Step 5: Export Ontology-Aligned RDF

`RDFExporter` writes the aligned graph as Turtle. Entity types (`rdf:type`) and relationship predicates now use the ontology URIs.

> [!NOTE]
> The file exporter serializes a compact projection of each entity (name and confidence); the complete property triples are produced by the triplet store in Step 6.


In [ ]:
from semantica.export import RDFExporter

exporter = RDFExporter()
exporter.export(aligned_graph, "semantic_layer.ttl", format="turtle")

turtle = open("semantic_layer.ttl").read()
print(turtle[:600])

# The RDF reflects the ontology mappings, not the raw source labels
assert BASE_URI + "Person" in turtle
assert BASE_URI + "worksFor" in turtle
print("... exported semantic_layer.ttl")

## Step 6: Store and Query

`TripletStore.store()` loads the aligned graph together with the ontology. The embedded Oxigraph backend runs in memory, so there is nothing to install or start beyond the `pip` extra. With the data in the store, a business question — *who works for Tech Corp, and in which role?* — becomes one small SPARQL query over the ontology's terms.


In [ ]:
from semantica.triplet_store import TripletStore

store = TripletStore(backend="oxigraph")
result = store.store(aligned_graph, ontology)
print(f"Stored triples: {result['processed']} (failed: {result['failed']})")

query = f"""
SELECT ?name ?role WHERE {{
    ?person <{BASE_URI}worksFor> ?org .
    ?person <{BASE_URI}name> ?name .
    ?person <{BASE_URI}role> ?role .
}}
ORDER BY ?name
"""
query_result = store.execute_query(query)

print("\nWho works for Tech Corp, and in which role?")
for binding in query_result.bindings:
    print(f"  {binding['name']['value']} — {binding['role']['value']}")

assert result["failed"] == 0
assert len(query_result.bindings) == 2

## 🧹 Optional: Clean Up


In [ ]:
from pathlib import Path

ttl_file = Path("semantic_layer.ttl")
if ttl_file.exists():
    ttl_file.unlink()
    print(f"Removed {ttl_file}")

## Summary

A minimal semantic layer is a composition, and you have now built each part:

1. **Knowledge graph** — `GraphBuilder` from explicit entities and relationships
2. **Ontology** — `OntologyGenerator` with your `base_uri`
3. **Explicit mappings** — entity types, relationship types, and properties, each tied to an ontology term
4. **Ontology-aligned RDF** — the mappings applied to the graph, exported with `RDFExporter`
5. **Queryable store** — `TripletStore` (embedded Oxigraph) answering a SPARQL question over the shared vocabulary

### Where to go next

The production version of this workflow — hand-designed governed ontologies, explicit source-to-ontology mappings from a warehouse, n-ary modeling, SHACL validation, provenance, and versioning — is covered in [Advanced: Manual Ontology + Snowflake Mapping](../advanced/13_Manual_Ontology_Snowflake_Mapping.ipynb).
